In [1]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 12.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [5]:

from google.colab import files
uploaded = files.upload()

Saving preprocessed_dataset.csv to preprocessed_dataset (1).csv


In [6]:
import pandas as pd
bc_data = pd.read_csv("preprocessed_dataset.csv")

In [7]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

target = "attack type"

train_df, test_df = train_test_split(
    bc_data,
    test_size=0.25,
    random_state=42,
    stratify=bc_data[target]
)

predictor = TabularPredictor(
    label=target,
    eval_metric="f1"
).fit(
    train_data=train_df,
    presets="high_quality",
    time_limit=1800
)

pred = predictor.predict(test_df)

print("Accuracy :", accuracy_score(test_df[target], pred))
print("Precision:", precision_score(test_df[target], pred))
print("Recall   :", recall_score(test_df[target], pred))
print("F1 Score :", f1_score(test_df[target], pred))

print("\nBest Model:")
print(predictor.info()["model_best"])

print("\nLeaderboard:")
print(predictor.leaderboard(test_df))

print("\nClassification Report:")
print(classification_report(test_df[target], pred))

print("\nConfusion Matrix:")
print(confusion_matrix(test_df[target], pred))

No path specified. Models will be saved in: "AutogluonModels/ag-20260729_202404"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.73 GB / 12.67 GB (84.7%)
Disk Space Avail:   58.66 GB / 112.64 GB (52.1%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will 

Accuracy : 0.9917333333333334
Precision: 0.9904761904761905
Recall   : 0.993103448275862
F1 Score : 0.9917880794701986

Best Model:


KeyError: 'model_best'

In [8]:
!pip install flaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 10.9 MB/s eta 0:00:00


In [10]:
from sklearn.model_selection import train_test_split
from flaml import AutoML
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


X = bc_data.drop(columns=[target])
y = bc_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

automl = AutoML()

settings = {
    "time_budget": 1800,
    "metric": "accuracy",      # or "macro_f1" if supported by your FLAML version
    "task": "classification",
    "seed": 42,
}

automl.fit(
    X_train=X_train,
    y_train=y_train,
    **settings
)

pred = automl.predict(X_test)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, average="weighted"))
print("Recall   :", recall_score(y_test, pred, average="weighted"))
print("F1 Score :", f1_score(y_test, pred, average="weighted"))

print("\nBest Model:", automl.best_estimator)
print("Best Config:", automl.best_config)

[flaml.automl.logger: 07-29 21:27:44] {2375} INFO - task = classification
[flaml.automl.logger: 07-29 21:27:44] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 07-29 21:27:44] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 07-29 21:27:44] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'catboost', 'lrl1']
[flaml.automl.logger: 07-29 21:27:44] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 07-29 21:27:45] {3046} INFO - Estimated sufficient time budget=9001s. Estimated necessary time budget=222s.
[flaml.automl.logger: 07-29 21:27:45] {3097} INFO -  at 1.0s,	estimator lgbm's best error=1.0800e-01,	best estimator lgbm's best error=1.0800e-01
[flaml.automl.logger: 07-29 21:27:45] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 07-29 21:27:46] {3097} INFO -  at 2.2s,	estimator lgbm's best error=1.0800e-01,	best estimator lgbm's best error=1.0800e-0

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 07-29 21:27:54] {3097} INFO -  at 10.6s,	estimator sgd's best error=2.8711e-01,	best estimator lgbm's best error=1.0800e-01
[flaml.automl.logger: 07-29 21:27:54] {2911} INFO - iteration 3, current learner lgbm
[flaml.automl.logger: 07-29 21:27:55] {3097} INFO -  at 11.6s,	estimator lgbm's best error=5.7156e-02,	best estimator lgbm's best error=5.7156e-02
[flaml.automl.logger: 07-29 21:27:55] {2911} INFO - iteration 4, current learner xgboost
[flaml.automl.logger: 07-29 21:27:57] {3097} INFO -  at 13.7s,	estimator xgboost's best error=1.0880e-01,	best estimator lgbm's best error=5.7156e-02
[flaml.automl.logger: 07-29 21:27:57] {2911} INFO - iteration 5, current learner extra_tree
[flaml.automl.logger: 07-29 21:27:59] {3097} INFO -  at 14.7s,	estimator extra_tree's best error=2.2480e-01,	best estimator lgbm's best error=5.7156e-02
[flaml.automl.logger: 07-29 21:27:59] {2911} INFO - iteration 6, current learner lgbm
[flaml.automl.logger: 07-29 21:28:02] {3097} INFO -

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 07-29 21:46:45] {3097} INFO -  at 1141.5s,	estimator lrl1's best error=8.8711e-02,	best estimator extra_tree's best error=7.5556e-03
[flaml.automl.logger: 07-29 21:46:45] {2911} INFO - iteration 207, current learner lrl1
[flaml.automl.logger: 07-29 21:46:49] {3097} INFO -  at 1145.3s,	estimator lrl1's best error=8.8533e-02,	best estimator extra_tree's best error=7.5556e-03
[flaml.automl.logger: 07-29 21:46:49] {2911} INFO - iteration 208, current learner sgd
[flaml.automl.logger: 07-29 21:46:49] {3097} INFO -  at 1145.5s,	estimator sgd's best error=7.2533e-02,	best estimator extra_tree's best error=7.5556e-03
[flaml.automl.logger: 07-29 21:46:49] {2911} INFO - iteration 209, current learner extra_tree
[flaml.automl.logger: 07-29 21:47:01] {3097} INFO -  at 1157.1s,	estimator extra_tree's best error=7.5556e-03,	best estimator extra_tree's best error=7.5556e-03
[flaml.automl.logger: 07-29 21:47:01] {2911} INFO - iteration 210, current learner sgd
[flaml.automl.logge